# Подключаем MCP к простому агенту

В [предыдущем ноутбуке](react-agent-from-scratch.ipynb) инструменты были Python-функциями внутри процесса агента. Теперь вынесем их в отдельный **MCP-сервер**, а агент получит их имена, описания и схемы аргументов по протоколу.

## Как связаны модель, агент и MCP

```text
Пользователь → агент ↔ LLM
                 │
             MCP-клиент
                 │ stdio: сообщения через stdin/stdout
             MCP-сервер
                 ├── get_all_flights
                 └── calculate_trip_cost
```

**Модель** выбирает инструмент и аргументы. **Агент** исполняет этот выбор через MCP-клиент, добавляет результат в историю и снова вызывает модель. **Сервер** хранит реализацию инструментов и ничего не знает о нашей LLM.

MCP — Model Context Protocol — задаёт общий способ обнаруживать и вызывать инструменты. Агент по-прежнему работает в цикле «модель → инструмент → модель»; меняется способ подключения инструментов. 

## 1. Установка


In [ ]:
!pip install "langchain[mcp]>=1.4,<2" "langchain-openai>=1,<2" vllm

### Поднимем LLM

```bash
vllm serve Qwen/Qwen3-4B-FP8 \
  --host 127.0.0.1 \
  --port 8000 \
  --served-model-name qwen \
  --max-model-len 8192 \
  --gpu-memory-utilization 0.85 \
  --enable-auto-tool-choice \
  --tool-call-parser hermes \
  --reasoning-parser qwen3
```


In [ ]:
from langchain_openai import ChatOpenAI

LLM_BASE_URL = "http://localhost:8000/v1"
LLM_MODEL = "qwen"
LLM_API_KEY = "EMPTY"

model = ChatOpenAI(
    base_url=LLM_BASE_URL,
    model=LLM_MODEL,
    api_key=LLM_API_KEY,
    temperature=0,
)

## 2. MCP-сервер в отдельном файле

`@mcp.tool` публикует функцию на сервере. Из type hints и docstring FastMCP строит схему аргументов и описание, которые затем увидит модель.

Два инструмента:

- `get_all_flights(date)` возвращает все рейсы на дату
- `calculate_trip_cost(ticket_price, hotel_price_per_night, nights)` считает бюджет: билет в одну сторону и проживание.

Код сервера находится в отдельном файле [mcp_flights_server.py](mcp_flights_server.py) рядом с ноутбуком. Откройте его в редакторе: там создаётся `FastMCP`, регистрируются инструменты и вызывается `mcp.run(transport="stdio")`. Менять реализацию инструментов можно прямо в этом файле. В `stdio` сервер запускает клиент: вручную запускать этот скрипт в терминале не нужно.


## 3. Подключаемся и получаем инструменты

`StdioTransport` запускает сервер отдельным процессом.

`MCPAdapter` превращает инструменты сервера в инструменты LangChain. `list_tools()` получает их по MCP — импортировать функции из серверного файла и вручную дублировать JSON-схемы не нужно.

### Как подключаются к MCP по сети

Если MCP-сервер уже запущен, а клиент подключается к нему URL, например `https://example.com/mcp`.

Чтобы попробовать HTTP на своем сервере:

1. В конце `mcp_flights_server.py` замените запуск на:

   ```python
   mcp.run(transport="http", host="127.0.0.1", port=8001)
   ```

2. В отдельном терминале того же Python-окружения запустите сервер и оставьте его работать:

   ```bash
   python mcp_flights_server.py
   ```

3. В следующей ячейке вместо подготовки `SERVER_PATH` и `StdioTransport(...)` задайте:

   ```python
   MCP_TRANSPORT = "http://127.0.0.1:8001/mcp"
   ```

Далее `MCPAdapter(MCP_TRANSPORT)`, обнаружение инструментов и агент работают так же.

In [ ]:
import json
import sys
from pathlib import Path

from fastmcp.client.transports import StdioTransport
from langchain.mcp import MCPAdapter

SERVER_PATH = Path("mcp_flights_server.py").resolve()
if not SERVER_PATH.is_file():
    raise FileNotFoundError(
        f"Не найден MCP-сервер: {SERVER_PATH}. "
        "Укажите путь к mcp_flights_server.py или откройте ноутбук из seminars/06-agents."
    )
# Для уже запущенного HTTP-сервера вместо SERVER_PATH и StdioTransport задайте:
# MCP_TRANSPORT = "http://127.0.0.1:8001/mcp"
MCP_TRANSPORT = StdioTransport(
    command=sys.executable,
    args=[str(SERVER_PATH)],
)


async def discover_tools():
    async with MCPAdapter(MCP_TRANSPORT) as adapter:
        return await adapter.list_tools()


tools = await discover_tools()
tools_by_name = {tool.name: tool for tool in tools}
for tool in tools:
    print(f"\n{tool.name}: {tool.description}")
    print(json.dumps(tool.args_schema, ensure_ascii=False, indent=2))


### Проверяем вызовы без модели

In [ ]:
import datetime as dt
from zoneinfo import ZoneInfo

from IPython.display import display

tomorrow = (dt.datetime.now(ZoneInfo("Europe/Moscow")).date() + dt.timedelta(days=1)).isoformat()

flights_result = await tools_by_name["get_all_flights"].ainvoke({"date": tomorrow})
display(flights_result)

budget_result = await tools_by_name["calculate_trip_cost"].ainvoke({
    "ticket_price": 3800,
    "hotel_price_per_night": 4500,
    "nights": 2,
})
display(budget_result)


Результат MCP может содержать текстовые блоки и структурированные данные. Адаптер подготавливает их для сообщений инструментов; вручную собирать `ToolMessage` здесь не требуется.

## 4. Передаём MCP-инструменты простому агенту

`create_agent` собирает готовый цикл на LangGraph: модель выбирает инструмент, агент выполняет его и передаёт результат модели. Вместо списка локальных `@tool` передаём полученный с сервера `tools`. Вызов `bind_tools` и сопоставление результатов с `tool_call_id` библиотека выполняет сама.

Агенту доступны описания и схемы инструментов, а их реализация остаётся на сервере.


In [ ]:
from langchain.agents import create_agent

today = dt.datetime.now(ZoneInfo("Europe/Moscow")).date().isoformat()
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        f"Ты помощник по учебным поездкам. Сегодня {today}, часовой пояс Europe/Moscow. "
        "Отвечай кратко и по-русски. Даты передавай в формате YYYY-MM-DD. "
        "Получай рейсы через get_all_flights и выбирай только нужное направление. "
        "Для бюджета используй calculate_trip_cost с ценой выбранного билета. "
        "Бюджет включает билет в одну сторону и отель для одного человека. "
        "Не выдумывай рейсы и цены. Если не хватает данных, задай уточняющий вопрос. "
        "Это учебные данные, бронирование не выполняется."
    ),
)


### Смотрим на агентный цикл

Выведем сообщения всех шагов: аргументы tool calls, результаты MCP-инструментов и итоговый ответ. Ожидаем вызов `get_all_flights`, затем `calculate_trip_cost` с ценой выбранного рейса. Число шагов ограничим, чтобы ошибка модели не приводила к бесконечному циклу.


In [ ]:
async def run_agent(query: str):
    print("Запрос пользователя:", query)
    async for update in agent.astream(
        {"messages": [{"role": "user", "content": query}]},
        config={"recursion_limit": 10},
        stream_mode="updates",
    ):
        for node_update in update.values():
            for message in node_update.get("messages", []):
                message.pretty_print()


await run_agent(
    f"Найди самый дешёвый рейс из Москвы в Санкт-Петербург на {tomorrow}. "
    "Я еду один, отель стоит 4500 рублей за ночь, останусь на 2 ночи. "
    "Посчитай общий бюджет билета в одну сторону и отеля. "
    "Укажи номер рейса и составляющие суммы."
)